# BudgetIndex — Reference Solution

For each document, choose at most `budget_k` non-overlapping windows of at most
60 tokens. A question is answered when one window holds **all** its evidence and
**none** of its planted false sentences.

The hard part is **not** a regex. A question and the sentence that answers it
word the attribute independently — a question may ask for the "number of
residents" while the answering sentence records a "resident headcount" — and
some wordings in the evaluation set never appear in training at all. So this
solution maps a wording to an attribute by **content-word overlap against
prototypes**, which generalises to wordings it has never seen. Referring
expressions are resolved structurally, not by template.

Then the true sentence is separated from its planted twin by reconciling the
value each states against `real_facts.json`, and a budgeted DP places the
windows.

Scores **0.6696** against a certified ceiling of **0.7021**.

In [ ]:
import re, json, os
import pandas as pd

PUBLIC = 'prepared/public'
MAX_CHUNK_TOKENS = 60

# Content words that characterise each attribute. Deliberately small: the point
# is that unseen wordings reuse these stems, not that every wording is listed.
ATTR_PROTOTYPES = {
    "population": {"population", "people", "resident", "residents", "inhabitant",
                   "inhabitants", "headcount", "populace", "living"},
    "area": {"area", "land", "landmass", "territory", "territorial", "expanse",
             "surface", "extent", "size", "km"},
    "num_borders": {"border", "borders", "bordering", "neighbour", "neighbours",
                    "neighbor", "neighbors", "neighbouring", "adjoining",
                    "adjacent", "frontier"},
    "num_languages": {"language", "languages", "tongue", "tongues", "official",
                      "linguistic", "recognised", "recognized", "designated"},
}

RE_WORD = re.compile(r"[a-z]+")
RE_CODE = re.compile(r"\b([A-Z]{3})\b")

RE_MILLION = re.compile(r"([\d,]+(?:\.\d+)?)\s*million", re.I)
RE_THOUSAND = re.compile(r"([\d,]+(?:\.\d+)?)\s*thousand", re.I)
RE_NUMBER = re.compile(r"\d[\d,]*(?:\.\d+)?")


def load_kb(public):
    with open(os.path.join(public, "real_facts.json"), encoding="utf-8") as f:
        rows = json.load(f)
    by_name = {r["name"]: r for r in rows}
    by_code = {r["cca3"]: r for r in rows}
    by_capital = {}
    for r in rows:
        by_capital.setdefault(r["capital"], r)
    return rows, by_name, by_code, by_capital


def stated_value(sentence):
    """The number a sentence denotes, handling scaled and grouped forms."""
    m = RE_MILLION.search(sentence)
    if m:
        return float(m.group(1).replace(",", "")) * 1e6
    m = RE_THOUSAND.search(sentence)
    if m:
        return float(m.group(1).replace(",", "")) * 1e3
    m = RE_NUMBER.search(sentence)
    if m:
        return float(m.group(0).replace(",", ""))
    return None


def attribute_of(text):
    """Which attribute does this wording denote? None if no prototype matches.

    Scored by how many characteristic content words the text shares with each
    prototype. This is what generalises to wordings never seen in training.
    """
    words = set(RE_WORD.findall(text.lower()))
    best, best_hits = None, 0
    for key, proto in ATTR_PROTOTYPES.items():
        hits = len(words & proto)
        if hits > best_hits:
            best, best_hits = key, hits
    return best


def resolve_entity(question, sents, by_name, by_code, by_capital):
    """(country_name, link_sentence_ids) or None.

    Structural, not template-driven: the question names a country, an ISO code,
    or a capital, and any linking sentences are recognised by what they contain
    rather than by matching a fixed phrasing.
    """
    # 1. the question names the country outright
    named = [n for n in by_name if n in question]
    if named:
        return max(named, key=len), []

    # 2. the question carries an ISO code -> find the sentence resolving it
    for code in RE_CODE.findall(question):
        if code not in by_code:
            continue
        for i, s in enumerate(sents):
            if code in s:
                here = [n for n in by_name if n in s]
                if here:
                    return max(here, key=len), [i]
        return None

    # 3. the question carries a capital
    caps = [c for c in by_capital if c in question]
    if not caps:
        return None
    cap = max(caps, key=len)
    # 3a. one hop: a sentence holding the capital and a country name
    for i, s in enumerate(sents):
        if cap in s:
            here = [n for n in by_name if n in s]
            if here:
                return max(here, key=len), [i]
    # 3b. two hops: capital -> ISO code, then code -> country
    for i, s in enumerate(sents):
        if cap not in s:
            continue
        for code in RE_CODE.findall(s):
            if code not in by_code:
                continue
            for j, t in enumerate(sents):
                if j == i or code not in t:
                    continue
                here = [n for n in by_name if n in t]
                if here:
                    return max(here, key=len), [i, j]
    return None


def predict(question, sents, by_name, by_code, by_capital):
    """(required_sent_ids, poison_sent_ids) or None."""
    attr = attribute_of(question)
    if attr is None:
        return None
    res = resolve_entity(question, sents, by_name, by_code, by_capital)
    if res is None:
        return None
    entity, link = res
    row = by_name.get(entity)
    if row is None:
        return None
    true_val = float(row[attr])

    cands = []
    for i, s in enumerate(sents):
        if entity not in s:
            continue
        if attribute_of(s) != attr:
            continue
        v = stated_value(s)
        if v is not None:
            cands.append((abs(v - true_val), i))
    if not cands:
        return None
    cands.sort()
    pick = cands[0][1]
    return sorted(set(link + [pick])), [i for _, i in cands[1:]]


def budget_windows(tok, queries, cap, K):
    """At most K non-overlapping <=cap-token windows maximising answered
    queries, with backtracking."""
    n = len(tok)
    pref = [0]
    for t in tok:
        pref.append(pref[-1] + t)
    qinfo = [(min(r), max(r), set(p)) for r, p in queries]

    def gain(j, i):
        return sum(1 for lo, hi, poi in qinfo
                   if j <= lo and hi < i and not any(j <= p < i for p in poi))

    g = [[0] * (n + 1) for _ in range(K + 1)]
    bk = [[None] * (n + 1) for _ in range(K + 1)]
    for k in range(1, K + 1):
        for i in range(1, n + 1):
            best, arg = g[k][i - 1], ("skip", i - 1)
            if g[k - 1][i] > best:
                best, arg = g[k - 1][i], ("drop", i)
            j = i - 1
            while j >= 0 and pref[i] - pref[j] <= cap:
                v = g[k - 1][j] + gain(j, i)
                if v > best:
                    best, arg = v, ("win", j)
                j -= 1
            g[k][i], bk[k][i] = best, arg
    spans, k, i = [], K, n
    while k > 0 and i > 0:
        kind, j = bk[k][i]
        if kind == "win":
            spans.append((j, i))
            k -= 1
            i = j
        elif kind == "drop":
            k -= 1
        else:
            i = j
    return sorted(spans)


In [ ]:
docs = pd.read_csv(os.path.join(PUBLIC, 'documents.csv'))
test = pd.read_csv(os.path.join(PUBLIC, 'queries_test.csv'))
_, by_name, by_code, by_capital = load_kb(PUBLIC)

sents_by, tok_by, bud = {}, {}, {}
for did, grp in docs.groupby('doc_id', sort=True):
    grp = grp.sort_values('sent_id')
    sents_by[did] = grp.text.tolist()
    tok_by[did]   = grp.n_tokens.tolist()
    bud[did]      = int(grp.budget_k.iloc[0])

pred = {d: [] for d in sents_by}
hit = 0
for r in test.itertuples(index=False):
    p = predict(r.question, sents_by[r.doc_id], by_name, by_code, by_capital)
    if p is not None:
        pred[r.doc_id].append(p); hit += 1
print(f'located evidence for {hit}/{len(test)} questions')

rows = []
for did in sents_by:
    spans = budget_windows(tok_by[did], pred[did], MAX_CHUNK_TOKENS, bud[did])
    assert len(spans) <= bud[did]
    assert all(sum(tok_by[did][a:b]) <= MAX_CHUNK_TOKENS for a, b in spans)
    assert all(spans[i][0] >= spans[i-1][1] for i in range(1, len(spans)))
    rows.append({'doc_id': did, 'windows': '|'.join(f'{a}-{b}' for a, b in spans)})

pd.DataFrame(rows).to_csv('submission.csv', index=False)
print('wrote submission.csv:', len(rows), 'documents')

## Self-check against the private answers (offline)

In [ ]:
from grade import grade
print('answerability =', round(grade('submission.csv', 'prepared/private/answers.csv'), 4))
print('certified ceiling = 0.7021')